# Lab 3 — End-to-End RAG Demo

This clean-run demo builds the local FAISS policy index, retrieves policy evidence for an at-risk customer, generates a policy-grounded retention offer, and reports the saved four-case evaluation-harness evidence. The final CSV is retained at `docs/lab3-evidence/rag_eval.csv` so it can be reviewed without Bedrock or vector-store access.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / 'rag').is_dir():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from rag.offer_rag import OfferRAG, POLICY_DIR

rag = OfferRAG(offline=True)
print(f'Repository root: {repo_root}')
print(f'Policy documents: {len(list(POLICY_DIR.glob("*.md")))}')
print(f'Chunked policy passages: {len(rag.chunks)}')
print('Mode: deterministic local embeddings + policy-safe offer template')

Repository root: C:\Users\camer\dev\cs401r-camchamb
Policy documents: 4
Chunked policy passages: 112
Mode: deterministic local embeddings + policy-safe offer template


In [2]:
test_case = {
    'customer_id': 'CUST-10000317',
    'customer_context': {'churn_probability': 0.81, 'loyalty_tier': 'Gold', 'days_since_last_purchase': 47, 'total_lifetime_value': 3120.40, 'avg_order_value': 186.25, 'category_diversity_score': 0.5, 'online_to_store_ratio': 0.72, 'top_categories': ['Outdoor', 'Footwear']},
    'question': 'Generate a personalized retention offer for this at-risk Gold member who shops mostly online for outdoor gear and footwear.',
}
retrieved = rag.retrieve(test_case['customer_context'], strategy='final', k=4)
print(f"Customer: {test_case['customer_id']} ({test_case['customer_context']['loyalty_tier']}; top categories: {', '.join(test_case['customer_context']['top_categories'])})")
print('Retrieved chunk IDs:', [chunk.id for chunk in retrieved])
for chunk in retrieved:
    print(f'- {chunk.id}: {chunk.text.splitlines()[0][:100]}')

Customer: CUST-10000317 (Gold; top categories: Outdoor, Footwear)
Retrieved chunk IDs: ['KB-FAQ-022-018', 'POL-SHP-007-006', 'POL-SHP-007-007', 'KB-FAQ-022-005']
- KB-FAQ-022-018: Shipping: **Is expedited shipping free for Gold or Platinum?**
- POL-SHP-007-006: 2. Free Shipping Thresholds: | Tier | Free standard shipping at |
- POL-SHP-007-007: 2. Free Shipping Thresholds: Free shipping applies to **standard** shipping only. Expedited and over
- KB-FAQ-022-005: Returns: **Do I pay for return shipping?**


In [3]:
result = rag.generate(test_case['customer_context'], test_case['question'], strategy='final')
print('Generated policy-grounded offer:')
print(result['answer'])
print(f'\nEvidence supplied to generation: {len(result["contexts"])} context items (customer profile + {len(result["chunk_ids"])} retrieved passages).')

Generated policy-grounded offer:
Return to NorthStar for Outdoor and make the most of your 1.5x points earn rate and free standard shipping. Your retention offer is single-use, tied to your account, and valid for 45 days; shop Outdoor when you are ready.

Evidence supplied to generation: 5 context items (customer profile + 4 retrieved passages).


In [4]:
import pandas as pd

# The harness creates a timestamped file in evaluation_results/. This stable copy is the
# rubric artifact and is deliberately committed with the notebook for offline TA review.
evaluation_path = repo_root / 'docs' / 'lab3-evidence' / 'rag_eval.csv'
timestamped_paths = sorted((repo_root / 'evaluation_results').glob('rag_eval_*.csv'))
assert evaluation_path.is_file(), f'Missing required evaluation artifact: {evaluation_path}'
assert timestamped_paths, 'No timestamped evaluation harness output found'
scores = pd.read_csv(evaluation_path)
print('Evaluation harness artifact:', evaluation_path.relative_to(repo_root))
print('Timestamped source artifact:', timestamped_paths[-1].relative_to(repo_root))
print(f'Cases evaluated: {len(scores)}')
for metric in ['faithfulness', 'answer_relevancy', 'context_recall']:
    print(f"Mean {metric.replace('_', ' ')}: {scores[metric].mean():.4f}")
print('\nPer-case RAGAS scores:')
print(scores[['faithfulness', 'answer_relevancy', 'context_recall']].to_string(index=False))

Evaluation harness artifact: docs\lab3-evidence\rag_eval.csv
Timestamped source artifact: evaluation_results\rag_eval_20261016T231246.csv
Cases evaluated: 4
Mean faithfulness: 0.4524
Mean answer relevancy: 0.3189
Mean context recall: 0.6833

Per-case RAGAS scores:
 faithfulness  answer_relevancy  context_recall
     0.500000          0.312366        1.000000
     0.166667          0.356584        0.400000
     0.571429          0.349124        0.333333
     0.571429          0.257466        1.000000
